In [93]:
import numpy as np
from scipy.optimize import minimize
from eval_utils import get_iou

def generate_utility_functions(num_agents, subissues_per_issue, total_utility, target_iou):
    """
    Generate utility functions for agents with constraints.
    """
    # Initialize agents with random scores
    agents = {
        f"agent_{i}": {
            "scores": {f"issue_{j}": np.random.rand(s) for j, s in enumerate(subissues_per_issue)},
            "threshold":np.random.normal(0.5, 0.1, 1)*100
        }
        for i in range(num_agents)
    }
    

    # Normalize initial scores to meet the total utility constraint
    for agent in agents.values():
        total = sum(np.sum(scores) for scores in agent["scores"].values())
        for issue, scores in agent["scores"].items():
            agent["scores"][issue] = (scores / total) * total_utility

    # Objective: Minimize deviation from the target IoU
    def objective(flat_scores):
        # Reshape flattened scores into agents' utility functions
        idx = 0
        for agent in agents.values():
            for issue, scores in agent["scores"].items():
                size = len(scores)
                agent["scores"][issue] = flat_scores[idx:idx + size]
                idx += size
        # Compute IoU
        avg_iou = get_iou(agents, use_numpy=True)
        return abs(avg_iou - target_iou)

    # Flatten the agents' utility scores for optimization
    flat_scores = np.concatenate([scores for agent in agents.values() for scores in agent["scores"].values()])

    # Constraints: Total utility per agent must equal total_utility, and all scores must be positive
    constraints = []
    idx = 0
    for agent in agents.values():
        max_sum_constraint = {
            'type': 'eq',
            'fun': lambda x, idx=idx, agent=agent: sum(
            np.max(x[idx + start : idx + end])
            for start, end in zip(
                [0] + list(np.cumsum([len(scores) for scores in agent["scores"].values()])[:-1]),
                np.cumsum([len(scores) for scores in agent["scores"].values()])
            )
        ) - 100
        }
        constraints.append(max_sum_constraint)
        idx += sum(len(scores) for scores in agent["scores"].values())

    # Non-negativity constraint
    bounds = [(0, None) for _ in range(len(flat_scores))]

    # Perform optimization
    result = minimize(
        objective,
        flat_scores,
        constraints=constraints,
        bounds=bounds,
        method='SLSQP',
        options={'maxiter': 1000, 'disp': True}
    )

    # Update agents with optimized scores
    idx = 0
    for agent in agents.values():
        for issue, scores in agent["scores"].items():
            size = len(scores)
            agent["scores"][issue] = result.x[idx:idx + size]
            idx += size

    return agents

In [94]:


# Example Usage
num_agents = 6
subissues_per_issue = [3, 3, 4, 4, 5]
total_utility = 100
target_iou = 0.53

agents = generate_utility_functions(num_agents, subissues_per_issue, total_utility, target_iou)

# Display Results
for agent_name, agent_data in agents.items():
    print(f"{agent_name}:")
    for issue, scores in agent_data["scores"].items():
        print(f"  {issue}: {scores}")
    print(f"  Threshold: {agent_data['threshold']}")
    print()



Optimization terminated successfully    (Exit mode 0)
            Current function value: 1.9353603275540365e-07
            Iterations: 14
            Function evaluations: 1618
            Gradient evaluations: 14
agent_0:
  issue_0: [ 5.81576486 20.37402952  4.49731002]
  issue_1: [ 6.13964947 19.94262722  6.70067612]
  issue_2: [ 5.78092335  7.02551681  4.4789559  20.42528314]
  issue_3: [ 6.08956437 20.39861397  4.42042258  6.11544125]
  issue_4: [ 4.07289621  3.26567176  5.75190493 18.85944616  4.76000511]
  Threshold: [55.778669]

agent_1:
  issue_0: [ 2.97906515  2.87421561 21.2479734 ]
  issue_1: [ 5.55552488  3.17883479 17.02255789]
  issue_2: [ 7.41614317  5.10264021  5.26288696 20.38799229]
  issue_3: [ 5.65469543 19.49400228  2.82148572  7.36868725]
  issue_4: [ 6.59043503  3.82389748 21.84747414  5.229833    2.02356143]
  Threshold: [48.35466768]

agent_2:
  issue_0: [ 5.60221973  4.6068686  19.67501149]
  issue_1: [ 6.49399384 19.17020449  7.21262522]
  issue_2: [21.3389

In [95]:
def check_sum_of_utilities(agents, total_utility):
    """
    Check if the sum of utilities for each agent equals the total utility.
    """
    for agent_name, agent_data in agents.items():
        total = sum(np.sum(np.max(scores)) for scores in agent_data["scores"].values())
        print(total)
        if not np.isclose(total, total_utility, atol=1e-6):
            print(f"Sum of utilities for {agent_name} is incorrect: {total} != {total_utility}")
            return False
    print("Sum of utilities for all agents is correct.")
    return True

def check_iou(agents, target_iou, tolerance=0.01):
    """
    Check if the IoU of the agents' scores is close to the target.
    """
    avg_iou = get_iou(agents, use_numpy=True)

    if abs(avg_iou - target_iou) > tolerance:
        print(f"Average IoU is outside tolerance: {avg_iou} != {target_iou}")
        return False
    print(f"Average IoU is within tolerance: {avg_iou} ≈ {target_iou}")
    return True

In [96]:
check_sum_of_utilities(agents, total_utility)
check_iou(agents, target_iou)

100.0
100.0
100.0
100.0
99.99999999999999
100.0
Sum of utilities for all agents is correct.
Average IoU is within tolerance: 0.5299998064639673 ≈ 0.53


True

In [97]:
def adjust_agent_scores(agents):
    """
    Floors all utility scores for each agent, calculates the difference between
    the total utility and 100, and adds the difference to the maximum score of issue_0.

    Parameters:
        agents (dict): Agent data containing scores for each issue.

    Returns:
        dict: Updated agent data with adjusted scores.
    """
    adjusted_agents = {}
    for agent_name, agent_data in agents.items():
        # Floor all scores
        floored_scores = {
            issue: np.floor(scores)
            for issue, scores in agent_data["scores"].items()
        }


        # Calculate the total utility
        total_utility = sum(np.sum(np.max(scores)) for scores in floored_scores.values())

        # Calculate the difference to 100
        utility_difference = 100 - total_utility
# 
        # Redistribute remaining utility mass randomly
        while utility_difference > 0:
            issue = f"issue_{np.random.randint(len(floored_scores))}"
            scores = floored_scores[issue]
            max_index = np.argmax(scores)
            scores[max_index] += 1
            floored_scores[issue] = scores
            utility_difference -= 1
        
        
        # if "issue_0" in floored_scores:
        #     issue_0_scores = floored_scores["issue_0"]
        #     max_index = np.argmax(issue_0_scores)  # Index of the maximum score
        #     issue_0_scores[max_index] += utility_difference
        #     floored_scores["issue_0"] = issue_0_scores

        # Update the agent's data
        adjusted_agents[agent_name] = {
            "scores": floored_scores,
            "threshold": np.floor(agent_data["threshold"]),  # Keep threshold unchanged
        }

    return adjusted_agents

In [98]:
floored_agents = adjust_agent_scores(agents)

# Display Results
for agent_name, agent_data in floored_agents.items():
    print(f"{agent_name}:")
    for issue, scores in agent_data["scores"].items():
        print(f"  {issue}: {scores}")
    print(f"  Threshold: {agent_data['threshold']}")

agent_0:
  issue_0: [ 5. 20.  4.]
  issue_1: [ 6. 19.  6.]
  issue_2: [ 5.  7.  4. 21.]
  issue_3: [ 6. 21.  4.  6.]
  issue_4: [ 4.  3.  5. 19.  4.]
  Threshold: [55.]
agent_1:
  issue_0: [ 2.  2. 22.]
  issue_1: [ 5.  3. 17.]
  issue_2: [ 7.  5.  5. 21.]
  issue_3: [ 5. 19.  2.  7.]
  issue_4: [ 6.  3. 21.  5.  2.]
  Threshold: [48.]
agent_2:
  issue_0: [ 5.  4. 20.]
  issue_1: [ 6. 19.  7.]
  issue_2: [21.  7.  4.  2.]
  issue_3: [ 6. 19.  2.  2.]
  issue_4: [ 2.  3.  7. 21.  4.]
  Threshold: [44.]
agent_3:
  issue_0: [ 6.  4. 21.]
  issue_1: [ 6. 20.  7.]
  issue_2: [ 3.  2. 18.  2.]
  issue_3: [21.  1.  5.  2.]
  issue_4: [ 4.  2.  2.  6. 20.]
  Threshold: [38.]
agent_4:
  issue_0: [ 5.  3. 20.]
  issue_1: [ 5. 19.  4.]
  issue_2: [20.  3.  2.  8.]
  issue_3: [ 6.  7.  6. 19.]
  issue_4: [ 1. 22.  5.  6.  6.]
  Threshold: [52.]
agent_5:
  issue_0: [ 5.  5. 19.]
  issue_1: [ 6. 21.  7.]
  issue_2: [ 2. 20.  5.  2.]
  issue_3: [ 3.  3. 19.  4.]
  issue_4: [ 3.  3.  2.  6. 21.]
  Thr

In [99]:
check_sum_of_utilities(floored_agents, total_utility)
check_iou(floored_agents, target_iou)

100.0
100.0
100.0
100.0
100.0
100.0
Sum of utilities for all agents is correct.
Average IoU is outside tolerance: 0.5006162246233149 != 0.53


False